# IT3051 - Fundamentals of Data Mining
## Used Car Price Prediction System
### Stage 6: Model Development & Stage 7: Model Optimization (XGBoost Regressor)

**Assigned Model:** Extreme Gradient Boosting (XGBoost Regressor)  
**Target Variable:** `log_price` (log-transformed selling price via `log1p`)  
**Evaluation Protocol:** 5-Fold Cross-Validation on Training Data (80%) + Final Generalization on Untouched Test Set (20%)

---
## 1. Setup & Preprocessing Integration
Loading the dataset and executing the exact preprocessing, feature engineering, and train/test split.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

pd.set_option('display.float_format', '{:,.4f}'.format)
np.random.seed(42)

# Load raw dataset
df = pd.read_csv('used_cars.csv')

# Clean raw currency and mileage strings
df['price']  = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['milage'] = df['milage'].str.replace(' mi.', '', regex=False).str.replace(',', '', regex=False).astype(float)

# Domain-guided missing value imputation
df_clean = df.copy()
df_clean['clean_title'] = df_clean['clean_title'].fillna('No')
df_clean['accident']    = df_clean['accident'].fillna('None reported')
electric_mask = df_clean['engine'].str.contains('Electric', case=False, na=False)
df_clean.loc[df_clean['fuel_type'].isnull() & electric_mask, 'fuel_type'] = 'Electric'
df_clean['fuel_type']   = df_clean['fuel_type'].fillna('Gasoline')

# Deduplication and Maserati typo removal ($2.95M data entry error)
df_clean = df_clean.drop_duplicates()
typo_mask = ((df_clean['brand'] == 'Maserati') & (df_clean['model_year'] == 2005) & (df_clean['price'] > 1_000_000))
df_clean = df_clean[~typo_mask].copy()

# Recode non-standard fuel types
valid_fuels = ['Gasoline', 'Hybrid', 'Electric', 'E85 Flex Fuel', 'Diesel', 'Plug-In Hybrid', 'Other']
df_clean['fuel_type'] = df_clean['fuel_type'].apply(lambda x: x if x in valid_fuels else 'Other')

# Target transformation: log(1 + price)
df_clean['log_price'] = np.log1p(df_clean['price'])

# Categorical Encodings
df_clean['is_clean_title'] = (df_clean['clean_title'] == 'Yes').astype(int)
df_clean['has_accident']   = (df_clean['accident'] == 'At least 1 accident or damage reported').astype(int)
df_clean['is_manual']      = df_clean['transmission'].str.contains('M/T|Manual', case=False, na=False).astype(int)

luxury_brands = ['Porsche', 'Bugatti', 'Ferrari', 'Lamborghini', 'Rolls-Royce', 'Bentley',
                 'McLaren', 'Maserati', 'Aston Martin', 'BMW', 'Mercedes-Benz', 'Audi',
                 'Lexus', 'Jaguar', 'Cadillac', 'Lincoln', 'Genesis', 'INFINITI',
                 'Tesla', 'Land Rover', 'Alfa Romeo', 'Volvo']
df_clean['is_luxury_brand'] = df_clean['brand'].isin(luxury_brands).astype(int)
fuel_dummies = pd.get_dummies(df_clean['fuel_type'], prefix='fuel', drop_first=True, dtype=int)

# Feature Engineering
df_clean['car_age'] = 2026 - df_clean['model_year']
df_clean['mileage_per_year'] = df_clean['milage'] / df_clean['car_age'].replace(0, 1)

# Feature Selection
df_final = pd.concat([df_clean, fuel_dummies], axis=1)
feature_cols = [
    'car_age', 'milage', 'mileage_per_year',
    'is_luxury_brand', 'is_manual', 'is_clean_title', 'has_accident'
] + list(fuel_dummies.columns)

X = df_final[feature_cols].copy()
y = df_final['log_price'].copy()

# 80/20 Train/Test Split (random_state=42)
shuffled_idx = np.random.permutation(len(X))
split_point  = int(0.8 * len(X))

X_train = X.iloc[shuffled_idx[:split_point]].copy()
X_test  = X.iloc[shuffled_idx[split_point:]].copy()
y_train = y.iloc[shuffled_idx[:split_point]].copy()
y_test  = y.iloc[shuffled_idx[split_point:]].copy()

# Leakage-Free Standardization
continuous_features = ['car_age', 'milage', 'mileage_per_year']
train_means = X_train[continuous_features].mean()
train_stds  = X_train[continuous_features].std()

X_train[continuous_features] = (X_train[continuous_features] - train_means) / train_stds
X_test[continuous_features]  = (X_test[continuous_features]  - train_means) / train_stds

print(f"Data Pipeline Verified:")
print(f"  Training set : {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"  Testing set  : {X_test.shape[0]} samples, {X_test.shape[1]} features")

---
## 2. Baseline XGBoost Regressor (Stage 6)
We evaluate an initial XGBoost model (`n_estimators=100`, `learning_rate=0.1`, `max_depth=6`) using 5-fold cross-validation strictly on `X_train`.

In [ ]:
# 5-Fold Cross-Validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_baseline = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

# Cross-validation
cv_metrics_base = cross_validate(
    xgb_baseline,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

base_cv_r2_mean   = cv_metrics_base['test_r2'].mean()
base_cv_r2_std    = cv_metrics_base['test_r2'].std()
base_cv_mae_mean  = -cv_metrics_base['test_mae'].mean()
base_cv_mae_std   = cv_metrics_base['test_mae'].std()
base_cv_rmse_mean = -cv_metrics_base['test_rmse'].mean()
base_cv_rmse_std  = cv_metrics_base['test_rmse'].std()

print("=== Baseline XGBoost (5-Fold CV on Training Set) ===")
print(f"Mean CV R²   : {base_cv_r2_mean:.4f} (+/- {base_cv_r2_std:.4f})")
print(f"Mean CV MAE  : {base_cv_mae_mean:.4f} (+/- {base_cv_mae_std:.4f})")
print(f"Mean CV RMSE : {base_cv_rmse_mean:.4f} (+/- {base_cv_rmse_std:.4f})")

---
## 3. Baseline Test Set Evaluation
Evaluating the baseline model on the untouched test set in both log-price space and original dollar space via `expm1`.

In [ ]:
# Fit on full training set
xgb_baseline.fit(X_train, y_train)

# Predict on test set
y_pred_base = xgb_baseline.predict(X_test)

base_test_r2   = r2_score(y_test, y_pred_base)
base_test_mae  = mean_absolute_error(y_test, y_pred_base)
base_test_rmse = root_mean_squared_error(y_test, y_pred_base)

# Price-scale metrics
y_test_dollars      = np.expm1(y_test)
y_pred_base_dollars = np.expm1(y_pred_base)

base_dollar_mae  = mean_absolute_error(y_test_dollars, y_pred_base_dollars)
base_dollar_rmse = root_mean_squared_error(y_test_dollars, y_pred_base_dollars)

print("=== Baseline XGBoost: Test Set Performance ===")
print(f"Log Scale R²       : {base_test_r2:.4f}")
print(f"Log Scale MAE      : {base_test_mae:.4f}")
print(f"Log Scale RMSE     : {base_test_rmse:.4f}")
print("-" * 50)
print(f"Dollar Scale MAE   : ${base_dollar_mae:,.2f}")
print(f"Dollar Scale RMSE  : ${base_dollar_rmse:,.2f}")

---
## 4. XGBoost Feature Importance
XGBoost feature importances reflect the fractional gain each feature provides to the tree branches.

In [ ]:
importances = pd.Series(xgb_baseline.feature_importances_, index=X_train.columns).sort_values(ascending=False)

importance_df = pd.DataFrame({
    'Feature': importances.index,
    'Gain Importance': importances.values,
    'Percentage (%)': (importances.values * 100).round(2)
})

print("=== Feature Importance Table ===")
display(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'][::-1], importance_df['Percentage (%)'][::-1], color='darkorange', edgecolor='black')
plt.title('XGBoost Feature Importances (Gain %)', fontsize=14, fontweight='bold')
plt.xlabel('Importance (%)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---
## 5. Hyperparameter Tuning (Stage 7: Optimization)
Tuning XGBoost hyperparameters with `RandomizedSearchCV` strictly on `X_train`:
- `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`

In [ ]:
param_distributions = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 4, 6],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1.0, 2.0]
}

search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=10,
    scoring='r2',
    cv=kf,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_hyperparams = search.best_params_

print("=== Best XGBoost Hyperparameters Found ===")
for k, v in best_hyperparams.items():
    print(f"  {k:18s}: {v}")
print(f"
Best Cross-Validation R²: {search.best_score_:.4f}")

---
## 6. Tuned XGBoost Evaluation
Evaluating the tuned model with 5-fold cross-validation and test set evaluation.

In [ ]:
xgb_tuned = XGBRegressor(**best_hyperparams, random_state=42, n_jobs=-1)

cv_metrics_tuned = cross_validate(
    xgb_tuned,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

tuned_cv_r2_mean   = cv_metrics_tuned['test_r2'].mean()
tuned_cv_r2_std    = cv_metrics_tuned['test_r2'].std()
tuned_cv_mae_mean  = -cv_metrics_tuned['test_mae'].mean()
tuned_cv_mae_std   = cv_metrics_tuned['test_mae'].std()
tuned_cv_rmse_mean = -cv_metrics_tuned['test_rmse'].mean()
tuned_cv_rmse_std  = cv_metrics_tuned['test_rmse'].std()

xgb_tuned.fit(X_train, y_train)
y_pred_tuned = xgb_tuned.predict(X_test)

tuned_test_r2   = r2_score(y_test, y_pred_tuned)
tuned_test_mae  = mean_absolute_error(y_test, y_pred_tuned)
tuned_test_rmse = root_mean_squared_error(y_test, y_pred_tuned)

y_pred_tuned_dollars = np.expm1(y_pred_tuned)
tuned_dollar_mae  = mean_absolute_error(y_test_dollars, y_pred_tuned_dollars)
tuned_dollar_rmse = root_mean_squared_error(y_test_dollars, y_pred_tuned_dollars)

print("=== Tuned XGBoost Performance ===")
print(f"Mean CV R²         : {tuned_cv_r2_mean:.4f} (+/- {tuned_cv_r2_std:.4f})")
print(f"Mean CV MAE        : {tuned_cv_mae_mean:.4f} (+/- {tuned_cv_mae_std:.4f})")
print(f"Mean CV RMSE       : {tuned_cv_rmse_mean:.4f} (+/- {tuned_cv_rmse_std:.4f})")
print("-" * 50)
print(f"Test R² (log)      : {tuned_test_r2:.4f}")
print(f"Test MAE (log)     : {tuned_test_mae:.4f}")
print(f"Test RMSE (log)    : {tuned_test_rmse:.4f}")
print(f"Test MAE ($ scale) : ${tuned_dollar_mae:,.2f}")
print(f"Test RMSE ($ scale): ${tuned_dollar_rmse:,.2f}")

---
## 7. Feature Selection Experiment
Testing feature subset reduction based on gain thresholds.

In [ ]:
threshold = 0.02
selected_features = importances[importances >= threshold].index.tolist()
dropped_features  = importances[importances < threshold].index.tolist()

print(f"Retained Features ({len(selected_features)}): {selected_features}")
print(f"Dropped Features  ({len(dropped_features)}): {dropped_features}")

X_train_selected = X_train[selected_features]
X_test_selected  = X_test[selected_features]

xgb_selected = XGBRegressor(**best_hyperparams, random_state=42, n_jobs=-1)

cv_metrics_sel = cross_validate(
    xgb_selected,
    X_train_selected,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

sel_cv_r2_mean   = cv_metrics_sel['test_r2'].mean()
sel_cv_r2_std    = cv_metrics_sel['test_r2'].std()
sel_cv_mae_mean  = -cv_metrics_sel['test_mae'].mean()
sel_cv_mae_std   = cv_metrics_sel['test_mae'].std()
sel_cv_rmse_mean = -cv_metrics_sel['test_rmse'].mean()
sel_cv_rmse_std  = cv_metrics_sel['test_rmse'].std()

xgb_selected.fit(X_train_selected, y_train)
y_pred_sel = xgb_selected.predict(X_test_selected)

sel_test_r2   = r2_score(y_test, y_pred_sel)
sel_test_mae  = mean_absolute_error(y_test, y_pred_sel)
sel_test_rmse = root_mean_squared_error(y_test, y_pred_sel)

y_pred_sel_dollars = np.expm1(y_pred_sel)
sel_dollar_mae  = mean_absolute_error(y_test_dollars, y_pred_sel_dollars)
sel_dollar_rmse = root_mean_squared_error(y_test_dollars, y_pred_sel_dollars)

print("=== Selected-Feature XGBoost Performance ===")
print(f"Mean CV R²         : {sel_cv_r2_mean:.4f} (+/- {sel_cv_r2_std:.4f})")
print(f"Mean CV MAE        : {sel_cv_mae_mean:.4f} (+/- {sel_cv_mae_std:.4f})")
print(f"Mean CV RMSE       : {sel_cv_rmse_mean:.4f} (+/- {sel_cv_rmse_std:.4f})")
print("-" * 50)
print(f"Test R² (log)      : {sel_test_r2:.4f}")
print(f"Test MAE (log)     : {sel_test_mae:.4f}")
print(f"Test RMSE (log)    : {sel_test_rmse:.4f}")
print(f"Test MAE ($ scale) : ${sel_dollar_mae:,.2f}")
print(f"Test RMSE ($ scale): ${sel_dollar_rmse:,.2f}")

---
## 8. Visual Diagnostics Dashboard
Visualizing actual vs. predicted values, residuals, and model comparison.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Actual vs Predicted
axes[0, 0].scatter(y_test, y_pred_tuned, alpha=0.35, color='darkorange', edgecolors='none')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal Fit (y=x)')
axes[0, 0].set_title('1. Actual vs Predicted Log Price (Tuned XGBoost)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Actual log(1 + price)')
axes[0, 0].set_ylabel('Predicted log(1 + price)')
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

# 2. Residuals vs Predicted
residuals = y_test - y_pred_tuned
axes[0, 1].scatter(y_pred_tuned, residuals, alpha=0.35, color='purple', edgecolors='none')
axes[0, 1].axhline(0, color='red', linestyle='--', lw=2)
axes[0, 1].set_title('2. Residuals vs Predicted Values', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Predicted log(1 + price)')
axes[0, 1].set_ylabel('Residual (Actual - Predicted)')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

# 3. Residual Distribution
axes[1, 0].hist(residuals, bins=35, color='darkcyan', edgecolor='black', alpha=0.8)
axes[1, 0].axvline(0, color='red', linestyle='--', lw=2)
axes[1, 0].set_title('3. Residuals Distribution (Centered around 0)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Residual Error')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

# 4. Model Comparison Bar Chart
models = ['Baseline XGB', 'Tuned XGB', 'Selected-Feature XGB']
r2_scores = [base_test_r2, tuned_test_r2, sel_test_r2]
rmse_scores = [base_test_rmse, tuned_test_rmse, sel_test_rmse]

x_pos = np.arange(len(models))
width = 0.35

axes[1, 1].bar(x_pos - width/2, r2_scores, width, label='Test R²', color='orange', edgecolor='black')
axes[1, 1].bar(x_pos + width/2, rmse_scores, width, label='Test RMSE', color='cornflowerblue', edgecolor='black')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(models)
axes[1, 1].set_title('4. XGBoost Model Comparison', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 9. Experiment Log & Ensemble Comparison

### XGBoost Experiments Summary
| Experiment | Model | Key Parameters | CV R² | Test R² | Test MAE (log) | Test RMSE (log) | Dollar MAE |
|---|---|---|:---:|:---:|:---:|:---:|:---:|
| **Exp 1** | **Baseline XGBoost** | 100 trees, depth=6, lr=0.1 | 0.6505 | 0.6425 | 0.3960 | 0.5114 | $17,941 |
| **Exp 2** | **Tuned XGBoost** | 200 trees, depth=3, lr=0.1, sub=1.0, col=0.85 | **0.6699** | **0.6625** | **0.3907** | **0.4968** | **$18,125** |
| **Exp 3** | **Selected-Feature XGB** | 13 features retained (all > 2%) | 0.6693 | 0.6611 | 0.3906 | 0.4978 | $18,150 |

---

### Grand Ensemble Benchmark: Random Forest vs. XGBoost

| Model Ensemble | Paradigm | Test R² | Test MAE (log) | Test RMSE (log) | Test MAE ($ scale) | Strengths / Trade-offs |
|---|---|:---:|:---:|:---:|:---:|---|
| **Tuned Random Forest** | **Bagging (Parallel)** | **0.6631** | **0.3891** | **0.4964** | **$17,410.57** | **Champion Dollar Accuracy:** Superior handling of high variance and luxury price outliers. |
| **Tuned XGBoost** | **Boosting (Sequential)** | 0.6625 | 0.3907 | 0.4968 | $18,125.33 | **Strong Regularization:** Faster inference, very competitive R² score. |